# Data Cleaning

Overall, my goal for data cleaning was to sort the 5 datasets I had downloaded (2 for traditional, 2 for RAPTOR, 1 for the past 5-year NBA playoffs) and merge them together into a final dataset that included individual and team playoff data from the 2019-2024 season.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
from sklearn.linear_model import LinearRegression, LogisticRegression
import duckdb
import time

#Read in 5 seperate CSV files
train=pd.read_csv("train.csv")
traditional=pd.read_csv("traditional.csv")
team_traditional=pd.read_csv("team_traditional.csv")
test=pd.read_csv("test.csv")
last5years=pd.read_csv("nba_playoffs_last_5_yr (2).csv")


   After reading the 5 CSV files, I started by sorting the traditional and team_traditional datasets. I converted both files' date columns to datetime by using the .to_datetime function, which allowed me to create a boolean mask that checked if the date was after the 2019 season. In addition, I also created a isplayoff mask which checked if the type column contained playoffs and the gameid contained 004 (004 indicates April, which is when the NBA playoffs are held). After sorting the data, I store it in a new intermediary CSV file. These were called "all_games_playoffs_2019plus.csv" and "all_team_games_playoffs_2019plus.csv". 

In [2]:
#traditional dataset sort to playoffs only and 2019 season onwards
traditional["date"] = pd.to_datetime(traditional["date"], errors="coerce")

isplayoff = traditional["type"].astype(str).str.contains\
("playoff", case=False, na=False) | \
            traditional["gameid"].astype(str).\
                str.startswith("004", na=False)

out = traditional[ isplayoff & (traditional["date"].dt.year >= 2019) ]
out.to_csv("all_games_playoffs_2019plus.csv", index=False)
playoffs2019=pd.read_csv("all_games_playoffs_2019plus.csv")

#team traditional sort to playoffs only and 2019 season onwards
team_traditional["date"] = pd.to_datetime(team_traditional["date"],\
                                          errors="coerce")


isplayoff = team_traditional["type"].astype(str).str.contains\
                ("playoff", case=False, na=False) | \
             team_traditional["gameid"].astype(str).\
                 str.startswith("004", na=False)

out = team_traditional[ isplayoff & (team_traditional\
    ["date"].dt.year >= 2019) ]
out.to_csv("all_team_games_playoffs_2019plus.csv", index=False)
teamplayoffs2019=pd.read_csv("all_team_games_playoffs_2019plus.csv")
teamplayoffs2019.head()


,gameid,date,type,teamid,team,home,away,MIN,PTS,FGM,...,DREB,REB,AST,TOV,STL,BLK,PF,+/-,win,season
0,41800111,2019-04-13,playoff,1610612753,ORL,TOR,ORL,48.0,104.0,36.0,...,38.0,48.0,19.0,11.0,9.0,4.0,19.0,3.0,1.0,2019
1,41800111,2019-04-13,playoff,1610612761,TOR,TOR,ORL,48.0,101.0,40.0,...,39.0,45.0,23.0,13.0,4.0,4.0,19.0,-3.0,0.0,2019
2,41800121,2019-04-13,playoff,1610612751,BKN,PHI,BKN,48.0,111.0,38.0,...,35.0,45.0,15.0,12.0,9.0,2.0,27.0,9.0,1.0,2019
3,41800121,2019-04-13,playoff,1610612755,PHI,PHI,BKN,48.0,102.0,35.0,...,34.0,50.0,20.0,13.0,6.0,11.0,24.0,-9.0,0.0,2019
4,41800141,2019-04-13,playoff,1610612744,GSW,GSW,LAC,48.0,121.0,45.0,...,42.0,54.0,31.0,21.0,9.0,14.0,22.0,17.0,1.0,2019


After sorting the two traditional datasets, I used an inner join to merge them into the joinedplayoffs2019 dataframe. The inner join was conducted on the gameid column to merge statistics from the same game together. 

In [3]:
#INNER JOIN which merges these two datasets together based on gameid
joinedplayoffs2019=duckdb.sql("SELECT * FROM playoffs2019\
            t INNER JOIN teamplayoffs2019 p\
            ON t.gameid=p.gameid").df()
joinedplayoffs2019.head()

,gameid,date,type,playerid,player,team,home,away,MIN,PTS,...,DREB_1,REB_1,AST_1,TOV_1,STL_1,BLK_1,PF_1,+/-_1,win_1,season_1
0,41800111,2019-04-13,playoff,203920,Khem Birch,ORL,TOR,ORL,15,6,...,39.0,45.0,23.0,13.0,4.0,4.0,19.0,-3.0,0.0,2019
1,41800111,2019-04-13,playoff,203487,Michael Carter-Williams,ORL,TOR,ORL,18,10,...,39.0,45.0,23.0,13.0,4.0,4.0,19.0,-3.0,0.0,2019
2,41800111,2019-04-13,playoff,201571,D.J. Augustin,ORL,TOR,ORL,30,25,...,39.0,45.0,23.0,13.0,4.0,4.0,19.0,-3.0,0.0,2019
3,41800111,2019-04-13,playoff,203932,Aaron Gordon,ORL,TOR,ORL,34,10,...,39.0,45.0,23.0,13.0,4.0,4.0,19.0,-3.0,0.0,2019
4,41800111,2019-04-13,playoff,1628371,Jonathan Isaac,ORL,TOR,ORL,40,11,...,39.0,45.0,23.0,13.0,4.0,4.0,19.0,-3.0,0.0,2019


Afterward, I moved on to the train dataset, which contained RAPTOR data. I first converted the "season_x" column to a number using the to_numeric function. I then sorted the dataset to only include dates from 2019 onwards, and then sorted the rows by ascending order. Finally, I stored the sorted train dataset into an intermediary CSV file named "train_2019_sorted.csv". 

In [4]:
#train to only include seasons from 2019 onwards
train["season_x"] = pd.to_numeric(train["season_x"], errors="coerce")
train = train[train["season_x"] >= 2019]
train = train.sort_values(by="season_x", ascending=True)  
train.to_csv("train_2019_sorted.csv", index=False)
train2019=pd.read_csv("train_2019_sorted.csv")
train2019.head()

,Unnamed: 0,player_name_x,player_id,season_x,poss_x,mp_x,raptor_offense_x,raptor_defense_x,raptor_total_x,war_total_x,...,raptor_offense,raptor_defense,raptor_total,war_total,war_reg_season,war_playoffs,predator_offense,predator_defense,predator_total,pace_impact
0,402990,Rajon Rondo,rondora01,2019,2982,1369,-0.703699,-1.392496,-2.096195,0.458531,...,2.154550,-1.585263,0.569287,0.536528,0.000000,0.536528,2.670823,-0.899685,1.771138,1.673826
1,135646,Langston Galloway,gallola01,2019,3868,1855,-0.067679,-0.824321,-0.892001,1.756106,...,-3.038095,1.981223,-1.056872,0.098368,0.000000,0.098368,-0.499610,0.470310,-0.029300,0.371627
2,127876,Bryn Forbes,forbebr01,2019,5148,2505,0.326588,-2.209900,-1.883312,1.123051,...,-4.551182,-4.587756,-9.138938,-0.920044,-0.920044,0.000000,-3.645838,-5.108243,-8.754082,-0.945298
3,52869,Bojan Bogdanovic,bogdabo02,2019,5622,2721,0.648770,-0.640112,0.008659,3.833364,...,-0.113616,-0.196089,-0.309704,3.080644,3.080644,0.000000,-0.452851,-0.371750,-0.824601,0.416100
4,996,Steven Adams,adamsst01,2019,6164,2828,-0.071750,1.875018,1.803267,6.601451,...,0.662443,1.660176,2.322620,4.329823,4.329823,0.000000,0.049831,1.948630,1.998460,-0.425098


The test dataset, which contained RAPTOR data, also needed to be sorted. I used the same approach to sort this one as I did for the train dataset. I first converted the "season_x" column to a number, and then sorted the dataset to only include seasons from 2019 onwards in ascending order. Finally, I stored the dataframe into an intermediary CSV file titled "test_2019_sorted.csv". 

In [5]:
#sort to only include seasons from 2019 onwards
test["season_x"] = pd.to_numeric(test["season_x"], errors="coerce")
test = test[test["season_x"] >= 2019]
test = test.sort_values(by="season_x", ascending=True)  
test.to_csv("test_2019_sorted.csv", index=False)
test2019=pd.read_csv("test_2019_sorted.csv")
test2019.head()

,Unnamed: 0,player_name_x,player_id,season_x,poss_x,mp_x,raptor_offense_x,raptor_defense_x,raptor_total_x,war_total_x,...,raptor_offense,raptor_defense,raptor_total,war_total,war_reg_season,war_playoffs,predator_offense,predator_defense,predator_total,pace_impact
0,249634,Reggie Jackson,jacksre01,2019,4941,2397,1.662452,-2.405671,-0.743219,2.440913,...,1.512500,-0.656411,0.856089,2.186055,2.186055,0.000000,0.961157,-0.838778,0.122380,-1.058409
1,343612,Paul Millsap,millspa01,2019,4839,2364,1.111536,2.755771,3.867307,8.016100,...,0.620935,2.227333,2.848268,6.757008,6.757008,0.000000,0.281613,2.591554,2.873167,0.950494
2,53484,Devin Booker,bookede01,2019,4772,2242,3.582047,-2.997283,0.584764,3.865201,...,3.267021,2.042009,5.309030,3.795195,0.000000,3.795195,4.135043,1.778557,5.913600,0.656855
3,70986,Jordan Clarkson,clarkjo01,2019,4575,2214,2.035009,-2.077628,-0.042619,3.042701,...,2.035009,-2.077628,-0.042619,3.042701,3.042701,0.000000,2.078466,-2.573353,-0.494886,-0.516888
4,392908,Duncan Robinson,robindu01,2019,341,161,-4.507935,-2.301672,-6.809607,-0.333990,...,0.543603,-0.526528,0.017075,3.179162,3.179162,0.000000,1.133695,-0.092088,1.041607,-0.450367


The last5years dataset is the one that includes offensive ratings for NBA teams that competed in the playoffs from 2019-2024. Since the dataset contains many columns, I want to only keep the columns that may be useful for my study. I first store the column names that I want into keepcols, and then set the last5years dataframe equal to the keepcols columns. I then renamed the "HOME_TEAM" and "AWAY_TEAM", and "HOME_PTS" and "AWAY_PTS" to "TEAM" and "PTS". This would allow me to vertically stack these columns so that each row represents a team's performance in a game, regardless of whether they are playing home or away.  

Next, I group by team and compute some 5-year statistics. These include the average offensive rating (team5yoffrating), average estimated offensive rating (team5yeoffrating), the mean number of points per game (team5ypts), and the number of games played (team5ygames). Finally, I store the new dataframe in an intermediary CSV file called "team_context_5y_aggregate.csv".

In [6]:
# Change rows from home and away to only include data for one team
keepcols = ["HOME_TEAM","AWAY_TEAM","HOME_PTS","AWAY_PTS","WL",
                         "OFF_RATING","E_OFF_RATING"]
             
last5years=last5years[keepcols].copy()


home = last5years.rename(columns={"HOME_TEAM":"TEAM","HOME_PTS":"PTS"})
away = last5years.rename(columns={"AWAY_TEAM":"TEAM","AWAY_PTS":"PTS"})
teamgames = pd.concat([home[["TEAM","PTS","OFF_RATING","E_OFF_RATING"]],
                        away[["TEAM","PTS","OFF_RATING","E_OFF_RATING"]]],
                       ignore_index=True)
#calculate  2019-2024 average statistics
teamagg = (teamgames.groupby("TEAM", as_index=False)
            .agg(team5yoffrating=("OFF_RATING","mean"),
                 team5yeoff_rating=("E_OFF_RATING","mean"),
                 team5ypts=("PTS","mean"),
                 team5ygames=("PTS","size")))

teamagg.to_csv("team_context_5y_aggregate.csv", index=False)
aggregate=pd.read_csv("team_context_5y_aggregate.csv")
aggregate.head()

,TEAM,team5yoffrating,team5yeoff_rating,team5ypts,team5ygames
0,Atlanta Hawks,112.313793,110.006897,106.724138,29
1,Boston Celtics,112.483529,110.409412,108.929412,85
2,Brooklyn Nets,115.245833,112.775000,107.458333,24
3,Chicago Bulls,97.600000,97.160000,95.200000,5
4,Cleveland Cavaliers,111.376471,109.847059,96.941176,17


Going back to the two RAPTOR datasets (train2019 and test2019), I combined these two dataframes using the .concat method. Afterward, I sorted by "season_x" (the season) and the player name by alphabetical order. I then dropped duplicates in case the train2019 and test2019 had some of the same players.  

In [7]:
#combine train and test datasets
raptor = pd.concat([train, test], ignore_index=True)
raptor = (raptor.sort_values(["season_x","player_name"])
                 .drop_duplicates(subset=\
                ["player_name","season_x"], keep="last"))
raptor.head()

,Unnamed: 0,player_name_x,player_id,season_x,poss_x,mp_x,raptor_offense_x,raptor_defense_x,raptor_total_x,war_total_x,...,raptor_offense,raptor_defense,raptor_total,war_total,war_reg_season,war_playoffs,predator_offense,predator_defense,predator_total,pace_impact
113246,148464,Aaron Gordon,gordoaa01,2019,5774,2797,0.054661,0.322631,0.377292,4.422695,...,-0.013502,0.713855,0.700352,4.616578,4.616578,0.000000,-0.285438,0.552131,0.266693,-0.406233
113673,205811,Aaron Holiday,holidaa01,2019,1418,659,-0.611643,2.205736,1.594093,1.464095,...,2.927518,-12.906263,-9.978745,-0.275572,0.000000,-0.275572,-0.154070,-10.734581,-10.888651,0.616925
114874,362777,Abdel Nader,naderab01,2019,1530,699,-4.399647,-2.311732,-6.711379,-1.408254,...,-3.482983,-0.017434,-3.500417,-0.331343,-0.331343,0.000000,-3.552393,-0.743282,-4.295675,-0.184311
115012,217171,Al Horford,horfoal01,2019,4796,2283,1.572570,2.193770,3.766339,7.618132,...,-0.903887,3.717163,2.813277,2.208856,2.208856,0.000000,-1.598748,3.060423,1.461674,-0.489514
114939,62900,Alec Burks,burksal01,2019,2850,1375,-0.870407,-1.599726,-2.470133,0.203663,...,-0.193446,-3.502867,-3.696312,-0.129826,-0.129826,0.000000,-0.041602,-2.647626,-2.689229,-0.038470


Because none of the datasets included columns for eFG% and TS%, I needed to use the data that was available to calculate these metrics. eFG% was calculated using the formula eFG%=(FGM+0.5x3PM)/FGA and TS% was calculated with the formula TS% = PTS / (2x(FGA + 0.44xFTA)). I then sorted the joinedplayoffs2019 dataset to only include the data I needed. These included "gameid", "date", "type", "season", "player", "team", "MIN", "PTS", "FGM", "FGA", "3PM", "FTA", "AST", "TOV", "eFG%", and "TS%". The reason I divided by .replace(0, np.nan) was in case a player took no shots, which means that it would be dividing by 0. In that case, the data cell would just become NaN instead of undefined. 

In [8]:
#eFG%=(FGM+0.5*3PM)/FGA
joinedplayoffs2019["eFG%"] = (joinedplayoffs2019["FGM"]\
                         + 0.5 * joinedplayoffs2019["3PM"])\
            / joinedplayoffs2019["FGA"].replace(0, np.nan)

# TS% = PTS / (2*(FGA + 0.44*FTA))
ts = (joinedplayoffs2019["FGA"] + 0.44 * joinedplayoffs2019["FTA"]) * 2
joinedplayoffs2019["TS%"] = joinedplayoffs2019["PTS"]\
                / ts.replace(0, np.nan)

keepcols = [
    "gameid","date","type","season","player","team",
    "MIN","PTS","FGM","FGA","3PM","FTA","AST","TOV",
    "eFG%","TS%"
]
joinedplayoffs2019 = joinedplayoffs2019[keepcols].copy()
joinedplayoffs2019 = joinedplayoffs2019.sort_values\
(["season","team","player","gameid"]).reset_index(drop=True)
joinedplayoffs2019 = joinedplayoffs2019.drop_duplicates().\
                        reset_index(drop=True)
joinedplayoffs2019.head()

,gameid,date,type,season,player,team,MIN,PTS,FGM,FGA,3PM,FTA,AST,TOV,eFG%,TS%
0,41800121,2019-04-13,playoff,2019,Caris LeVert,BKN,23,23,8,18,3,4,2,0,0.527778,0.581984
1,41800122,2019-04-15,playoff,2019,Caris LeVert,BKN,20,13,3,8,2,5,1,1,0.500000,0.637255
2,41800123,2019-04-18,playoff,2019,Caris LeVert,BKN,28,26,10,17,3,3,2,2,0.676471,0.709607
3,41800124,2019-04-20,playoff,2019,Caris LeVert,BKN,42,25,9,18,3,9,6,3,0.583333,0.569217
4,41800125,2019-04-23,playoff,2019,Caris LeVert,BKN,31,18,6,12,1,8,4,4,0.541667,0.579897


Before merging all three datasets, I cleaned the RAPTOR dataset one final time. I first cut it down to the columns I needed: "player_name", "season", "raptor_offense", "raptor_defense", "raptor_total", "war_total". I then standardized the player name by lowercasing and stripping spaces, converted the "season" column to a number, sorted by season, and dropped duplicate players. I also renamed the "player_name" column to "player" so that I could merge it with the other datasets more easily during the final merge. 

In [9]:
#raptor sorting and filtering

keepcols = ["player_name","season","raptor_offense",\
            "raptor_defense","raptor_total","war_total"]
raptor = raptor[keepcols].copy()
raptor["player_name_std"] = (raptor["player_name"].astype(str)
                         .str.lower().str.replace(r"[^\w\s]",""\
                            , regex=True).str.strip())
raptor["season"] = pd.to_numeric(raptor["season"], errors="coerce")
raptor = (raptor.sort_values(["season","player_name_std"])
        .drop_duplicates(subset=["player_name_std","season"],\
                         keep="last"))
raptor = raptor.sort_values(["season","player_name_std"]).\
    reset_index(drop=True)
raptor = raptor.rename(columns={"player_name": "player"})
raptor.head()



,player,season,raptor_offense,raptor_defense,raptor_total,war_total,player_name_std
0,Alex Len,2014,-4.290994,-5.106582,-9.397576,-1.235190,alex len
1,Blake Griffin,2014,3.689015,0.769263,4.458278,1.789044,blake griffin
2,Bradley Beal,2014,0.296486,0.433575,0.730061,4.480124,bradley beal
3,Cody Zeller,2014,-1.350666,-0.216940,-1.567606,0.861539,cody zeller
4,DeAndre Jordan,2014,-0.148232,0.936428,0.788196,0.819381,deandre jordan


Finally, I merged the three datasets into one. I used a pandas left join to merge the joinedplayoffs2019 and RAPTOR dataset and stored it in a dataframe called "merged". However, I was unable to merge "merged" with aggregate because they did not share any common columns. This is because for "aggregate", the team name was spelled out, while for "merged", the team name was abbreviated. I created a to_abbr function, which helped convert the spelled-out teams to their abbreviated form. I also renamed the "TEAM" column in the aggregate dataset to "team" so that it was consistent with the "merged" dataset. Finally, I was able to use a pandas left join to merge the "aggregate" and "team" datasets. 

In [10]:
#change spelled out team names to abbreviated
merged = joinedplayoffs2019.merge(raptor, how="left",\
                                on=["player", "season"])
TOABBR = {
    "atlanta hawks":"ATL","boston celtics":"BOS","brooklyn nets":"BKN","charlotte hornets":"CHA",
    "chicago bulls":"CHI","cleveland cavaliers":"CLE","detroit pistons":"DET","indiana pacers":"IND",
    "miami heat":"MIA","milwaukee bucks":"MIL","new york knicks":"NYK","orlando magic":"ORL",
    "philadelphia 76ers":"PHI","toronto raptors":"TOR","washington wizards":"WAS",
    "dallas mavericks":"DAL","denver nuggets":"DEN","golden state warriors":"GSW","houston rockets":"HOU",
    "la clippers":"LAC","los angeles clippers":"LAC","los angeles lakers":"LAL",
    "memphis grizzlies":"MEM","minnesota timberwolves":"MIN","new orleans pelicans":"NOP",
    "oklahoma city thunder":"OKC","phoenix suns":"PHX","portland trail blazers":"POR",
    "sacramento kings":"SAC","san antonio spurs":"SAS","utah jazz":"UTA"
}
def to_abbr(name):
    if name is None:
        return None
    s = str(name).strip()
    if len(s) == 3 and s.upper() == s:
        return s
    s2 = s.lower()
    for ch in [",", ".", "-", "_", "/", "(", ")", "'"]:
        s2 = s2.replace(ch, " ")
    s2 = " ".join(s2.split()) 
    if s2 in TOABBR:
        return TOABBR[s2]
    else:
        return None

aggregate = aggregate.rename(columns={"TEAM": "team"})
converted = []
for val in aggregate["team"]:
    converted.append(to_abbr(val))
aggregate["team"] = converted

merged = merged.merge(aggregate, how="left", on=["team"])
merged.to_csv("finalmerge.csv", index=False)
finalmerge=pd.read_csv("finalmerge.csv",dtype=str, low_memory=False)
finalmerge.head()


,gameid,date,type,season,player,team,MIN,PTS,FGM,FGA,...,TS%,raptor_offense,raptor_defense,raptor_total,war_total,player_name_std,team5yoffrating,team5yeoff_rating,team5ypts,team5ygames
0,41800121,2019-04-13,playoff,2019,Caris LeVert,BKN,23,23,8,18,...,0.5819838056680161,NaN,NaN,NaN,NaN,NaN,115.24583333333334,112.775,107.45833333333331,24.0
1,41800122,2019-04-15,playoff,2019,Caris LeVert,BKN,20,13,3,8,...,0.6372549019607844,NaN,NaN,NaN,NaN,NaN,115.24583333333334,112.775,107.45833333333331,24.0
2,41800123,2019-04-18,playoff,2019,Caris LeVert,BKN,28,26,10,17,...,0.7096069868995633,NaN,NaN,NaN,NaN,NaN,115.24583333333334,112.775,107.45833333333331,24.0
3,41800124,2019-04-20,playoff,2019,Caris LeVert,BKN,42,25,9,18,...,0.5692167577413478,NaN,NaN,NaN,NaN,NaN,115.24583333333334,112.775,107.45833333333331,24.0
4,41800125,2019-04-23,playoff,2019,Caris LeVert,BKN,31,18,6,12,...,0.5798969072164949,NaN,NaN,NaN,NaN,NaN,115.24583333333334,112.775,107.45833333333331,24.0


I realized that I forgot to calculate the USG% column, so I wrote code that aggregated team total stats such as teamFGA, teamFTA, teamTOV, and teamMIN. I merged these new stat columns with the final merge dataset using a left join. I then calculated the USG% using the formula USG%= 100 * ((FGA + 0.44 * FTA + TOV) * (TeamMinutes / 5))/(PlayerMinutes * (TeamFGA + 0.44 * TeamFTA + TeamTOV)). 

Additionally, I realized that for a lot of the players, there was no available RAPTOR information for them. Therefore, I dropped all the rows that had NaN for the raptor columns. Finally, I stored the "finalmerge" dataframe into a new intermediary CSV "finalmerge_usg.csv". 

In [11]:
#add aggregate total stat columns
for i in ["FGA","FTA","TOV","MIN"]:
    finalmerge[i] = pd.to_numeric(finalmerge[i], errors="coerce")

teamtotals = (finalmerge.groupby(["gameid","team"], as_index=False)
               .agg(teamFGA=("FGA","sum"), teamFTA=("FTA","sum"),
                    teamTOV=("TOV","sum"), teamMIN=("MIN","sum")))
finalmerge = finalmerge.merge(teamtotals, on=["gameid","team"], how="left")

# USG% = 100 * ((FGA + 0.44 * FTA + TOV) * (TeamMinutes / 5)) /
#(PlayerMinutes * (TeamFGA + 0.44 * TeamFTA + TeamTOV))
num = (finalmerge["FGA"] + 0.44*finalmerge["FTA"]\
       + finalmerge["TOV"]) * (finalmerge["teamMIN"]/5.0)
den = finalmerge["MIN"].replace(0, np.nan) * \
(finalmerge["teamFGA"] + 0.44*finalmerge["teamFTA"]\
 + finalmerge["teamTOV"]).replace(0, np.nan)
finalmerge["USG%"] = 100 * (num / den)

finalmerge = finalmerge.dropna(subset=["raptor_total",\
                                       "raptor_offense", "raptor_defense"])
finalmerge.to_csv("finalmerge_usg.csv", index=False)
finalmerge.head()

,gameid,date,type,season,player,team,MIN,PTS,FGM,FGA,...,player_name_std,team5yoffrating,team5yeoff_rating,team5ypts,team5ygames,teamFGA,teamFTA,teamTOV,teamMIN,USG%
5,41800121,2019-04-13,playoff,2019,D'Angelo Russell,BKN,29,26,10,25,...,dangelo russell,115.24583333333334,112.775,107.45833333333331,24.0,88,26,11,241,46.954502
6,41800122,2019-04-15,playoff,2019,D'Angelo Russell,BKN,25,16,6,16,...,dangelo russell,115.24583333333334,112.775,107.45833333333331,24.0,90,29,14,239,33.471463
7,41800123,2019-04-18,playoff,2019,D'Angelo Russell,BKN,30,26,12,27,...,dangelo russell,115.24583333333334,112.775,107.45833333333331,24.0,96,35,15,239,37.816456
8,41800124,2019-04-20,playoff,2019,D'Angelo Russell,BKN,37,21,6,19,...,dangelo russell,115.24583333333334,112.775,107.45833333333331,24.0,90,31,14,240,25.584238
9,41800125,2019-04-23,playoff,2019,D'Angelo Russell,BKN,27,8,3,16,...,dangelo russell,115.24583333333334,112.775,107.45833333333331,24.0,93,25,14,240,26.937853


# Errors Within Merged Dataset and Fixing it

I conducted a final sort for my "finalmerge" dataset. 
- I checked if there were duplicate game IDs and players and dropped them.
- I made sure the "MIN", "FGA", "TS%", and "eFG%" were all numbers. 
- I removed players who played less than 5 minutes and who did not attempt any field goals (shots)
- I conducted a final filtering of the columns, keeping the ones that I thought would be helpful for my analysis.

Once everything was sorted and complete, I stored the "finalmerge" dataset into a CSV file called "finalmerge_usg.csv". 

In [12]:
#final merge changes to columns and removing data that may not be useful
finalmerge = finalmerge.drop_duplicates(subset=["gameid", "player"])
for i in ["MIN", "FGA", "TS%", "eFG%"]:
    finalmerge[i] = pd.to_numeric(finalmerge[i], errors="coerce")
finalmerge = finalmerge[finalmerge["MIN"] >= 5]
finalmerge= finalmerge[finalmerge["FGA"] > 0] 
finalmerge= finalmerge.dropna(subset=["TS%", "eFG%"])
keepcols = [
    "gameid", "date", "season", "type", "player", "team",
    "MIN", "FGA", "FTA", "TOV", "PTS", "USG%", "TS%", "eFG%",
    "raptor_offense", "raptor_defense", "raptor_total", "war_total",
    "team5yoffrating", "team5yeoff_rating",\
    "team5y_pts", "team5y_games"
]
keepcolspresent = []
for i in keepcols:
    if i in finalmerge.columns:
        keepcolspresent.append(i)
finalmerge = finalmerge[keepcolspresent].copy()
finalmerge.to_csv("finalmerge_usg.csv", index=False)
finalmerge.head()


,gameid,date,season,type,player,team,MIN,FGA,FTA,TOV,PTS,USG%,TS%,eFG%,raptor_offense,raptor_defense,raptor_total,war_total,team5yoffrating,team5yeoff_rating
5,41800121,2019-04-13,2019,playoff,D'Angelo Russell,BKN,29,25,5,4,26,46.954502,0.477941,0.440000,2.735093554,-0.576171021,2.158922533,6.180959451,115.24583333333334,112.775
6,41800122,2019-04-15,2019,playoff,D'Angelo Russell,BKN,25,16,1,4,16,33.471463,0.486618,0.468750,2.735093554,-0.576171021,2.158922533,6.180959451,115.24583333333334,112.775
7,41800123,2019-04-18,2019,playoff,D'Angelo Russell,BKN,30,27,0,3,26,37.816456,0.481481,0.481481,2.735093554,-0.576171021,2.158922533,6.180959451,115.24583333333334,112.775
8,41800124,2019-04-20,2019,playoff,D'Angelo Russell,BKN,37,19,5,2,21,25.584238,0.495283,0.421053,2.735093554,-0.576171021,2.158922533,6.180959451,115.24583333333334,112.775
9,41800125,2019-04-23,2019,playoff,D'Angelo Russell,BKN,27,16,2,1,8,26.937853,0.236967,0.218750,2.735093554,-0.576171021,2.158922533,6.180959451,115.24583333333334,112.775


In [13]:
dataset=pd.read_csv("finalmerge_usg.csv")
dataset.head()
#filter out players who played less than 50 minutes in the playoffs
totalminutes=50
dataset["total_MIN_played"] = dataset.groupby(["player", "season"])["MIN"].transform("sum")
dataset = dataset[dataset["total_MIN_played"] >= 50].copy()


To make my dataset more appropriate for testing my hypotheses, I had to make some minor tweaks to it. To continue cleaning the dataset, I filtered out players who did not play more than 50 minutes in the entire playoffs because they could skew the data. Metrics such as TS% and RAPTOR impact can vary heavily depending on the possession count. Therefore, if a player participated in too few possessions, making a shot or two could widely swing the stats, and ruin the relationship I am trying to measure. 

In [14]:
#filter out players with usg%<5 because they are likely not offensive players
dataset = dataset.dropna(subset=["USG%", "TS%", "raptor_total"]).copy()
dataset = dataset[dataset["USG%"] >= 5].copy()

I also filtered out players with a USG% <5 for similar reasons as to why I filtered out players who played less than 50 total minutes in the playoffs. Those who played less than 5 minutes likely do not have a great offensive role in the team, and most likely only played in garbage time or injury minutes. These players generally:

- take almost zero shots

- never create offense

- are typically defensive specialists or deep bench players

Therefore, including them in my dataset creates a few problems:

- Problem 1: Their TS% is artificially inflated since low-usg players get the easiest shots, such as cuts, putbacks, and uncontested corner threes. This makes TS% higher at a lower usage, which can create a misleading result when I analyze the relationship between usage and efficiency. 

- Problem 2: They don't really reflect my research question. Because these players aren't really offensive players, they aren't really relevant to my question of how increased offensive usage affects a player's efficiency. 



In [15]:
#Filter out bottom 1 percentile and top 1 percentile
#Create dummies for OFF_RTG column
dataset["USG"] = dataset["USG%"]
dataset["TS"] = dataset["TS%"].astype(float)

if "team5yoffrating" in dataset.columns:
    playoff_avg_ortg = dataset["team5yoffrating"].mean()
    dataset["OffStr"] = (dataset["team5yoffrating"]\
                        > playoff_avg_ortg).astype(int)
    dataset = pd.get_dummies(dataset, columns=["OffStr"], drop_first=True)
dataset.to_csv("dataset_phase5.csv", index=False)

Finally, I create a dummy variable for a team's offensive strength based on whether or not their "Off_RTG" was higher than the league average. If it is higher, then they are assigned a 1. If not, then they are assigned a value of 0. This dummy variable will be useful when I use a multivariable regression to conduct my first hypothesis. 